# Lightweight U-Net v2 — Colab

Swin v2と同じ6バンド対応・衛星別補正・location非重複5-foldを使う軽量モデルです。fold別予測も保存するため、後からSwinとの重み付きアンサンブルができます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q rasterio
!nvidia-smi | head -5

In [ ]:
from pathlib import Path
import os
import sys

# TIFFはGoogle Driveから直接読むと極端に遅いため、データはColabローカルを使う。
# /content/train_dataset 等がない場合は、先にZIPを/contentへ展開すること。
ROOT = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/solafune-workspace')
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

for required in ('train_dataset', 'evaluation_dataset', 'sample_submission'):
    assert (ROOT / required).is_dir(), f'Not found: {ROOT / required}'

# 学習データはローカル、チェックポイントだけDriveへ永続化する。
drive_models = DRIVE_ROOT / 'models'
drive_models.mkdir(parents=True, exist_ok=True)
local_models = ROOT / 'models'
if local_models.is_symlink() and local_models.resolve() != drive_models.resolve():
    local_models.unlink()
if not local_models.exists():
    local_models.symlink_to(drive_models, target_is_directory=True)

from swin_nowcast_v2 import Config, create_submission, get_device, make_folds, prepare_metadata
from unet_nowcast_v2 import ensemble_unet_predict, train_unet_fold

device = get_device()
config = Config(
    root=str(ROOT),
    encoder_size=96,
    batch_size=32,       # OOMなら16または8
    epochs=15,
    lr_head=3e-4,
    workers=4,
    pretrained=False,
    use_amp=True,
    band_mode='matched6',
    use_satellite_normalization=True,
    unet_model_subdir='unet_v2',
)
train_df = prepare_metadata(config.train_dir / 'train_dataset.csv')
eval_df = prepare_metadata(config.evaluation_dir / 'evaluation_target.csv')
folds = make_folds(train_df, config.n_folds)
print(device, len(train_df), len(eval_df))

## 学習

最初は`TRAIN_FOLDS = [0]`、`epochs=1`で動作確認してください。本学習では5foldを指定します。

In [ ]:
TRAIN_FOLDS = list(range(5))
results = []
for fold_index in TRAIN_FOLDS:
    result = train_unet_fold(config, train_df, folds[fold_index], device=device, base_channels=32)
    results.append(result)

print({r['fold']: r['validation_rmse'] for r in results})
print('Mean CV:', sum(r['validation_rmse'] for r in results) / len(results))

## fold別推論・U-Net提出ZIP

`models/unet_v2/predictions_fold*.npy`も保存されます。

In [ ]:
unet_predictions = ensemble_unet_predict(config, eval_df, list(range(5)), device=device)
print(unet_predictions.shape, unet_predictions.min(), unet_predictions.mean(), unet_predictions.max())

unet_zip = DRIVE_ROOT / 'submission_unet_v2.zip'
create_submission(config, eval_df, unet_predictions, unet_zip)
print(unet_zip)

## Swin提出ZIPとのアンサンブル

Swinの提出ZIPを直接読み、同じ行順で混合します。まずはSwin 0.7 / U-Net 0.3が安全です。

In [ ]:
import zipfile
import numpy as np
from rasterio.io import MemoryFile

def read_submission_predictions(zip_path, evaluation_frame):
    arrays = []
    with zipfile.ZipFile(zip_path) as archive:
        for filename in evaluation_frame['gpm_imerg_filename']:
            with MemoryFile(archive.read(f'test_files/{filename}')) as memory_file:
                with memory_file.open() as src:
                    arrays.append(src.read(1).astype(np.float32))
    return np.stack(arrays)

swin_zip = DRIVE_ROOT / 'submission_swin_v2.zip'  # 実際のファイル名に合わせる
swin_predictions = read_submission_predictions(swin_zip, eval_df)

SWIN_WEIGHT = 0.7
blend = SWIN_WEIGHT * swin_predictions + (1 - SWIN_WEIGHT) * unet_predictions
blend_zip = DRIVE_ROOT / 'submission_swin70_unet30.zip'
create_submission(config, eval_df, blend, blend_zip)
print(blend_zip)